Due to the current situation (`Updated 09/02/2026`),

- Google Gemini has reduced the rate limits for several models, such as `gemini-2.5-flash` and `gemini-3-flash` (text models used in Colab notebooks), to a **limit of 20 Requests Per Day (RPD)**.

- To continue using these models seamlessly with sufficient rate limits, it is necessary to upgrade to the **pay-as-you-go tier** (link a Billing Account).
  - 👉 You can learn how to do this here: [https://ai.google.dev/gemini-api/docs/billing](https://ai.google.dev/gemini-api/docs/billing)

- Alternatively, you can follow the Groq API approach described below.

- Update `23/08/2026`: Removed the `llama` models because they are no longer supported by Groq, and replaced them with `qwen/qwen3.6-27b`.


Most of concepts and codes are adapted from this [repo](https://github.com/dair-ai/Prompt-Engineering-Guide).

Implementation Detail:
- LangChain is used.

# Setting environments and model setup

In [1]:
from IPython.display import display, Markdown

## Approach 1: Gemini

In [ ]:
# %%capture
# !pip install -qU langchain-google-genai

Request for Google API KEY here : https://aistudio.google.com/app/apikey

In [ ]:
# from getpass import getpass
# import os

# if "GOOGLE_API_KEY" not in os.environ:
#     os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google AI API key: ")

Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     temperature=1,
#     max_tokens=None,
#     timeout=None,
#     max_retries=2,
#     # other params...
# )

## Approach 2: Groq API


However, we still have an **alternative** that can be used via a free-tier API: **Groq API** (compatible with LangChain). This does not require linking a credit card and offers several models, such as:

Available models: https://console.groq.com/settings/limits

👉 You can sign up and get your API Key here: [https://console.groq.com/keys](https://console.groq.com/keys)


In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.9 MB/s eta 0:00:00


In [3]:
import getpass
import os

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


Since this is an open-ended generation, we do not want the generation to be boring so temperature is set to 1 instead of 0.

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    reasoning_effort="none",
    max_tokens=1000,
    timeout=None,
    max_retries=2,
)

# Zero-shot / Few-shot Prompting

**Ex. Summarize the customer feedback.**

In [5]:
# Zero-shot

prompt = """Summarize the customer feedback.
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:"""

zero_shot = llm.invoke(prompt)
display(Markdown(zero_shot.content))

The customer found the room comfortable with a good view but was unable to sleep due to a noisy air conditioner. Despite contacting reception, no technician arrived to address the issue, leading to a complaint about the hotel's slow response to maintenance requests.

The output is in a free-form format. If you want the model to follow a specific structure, you can:
- Add output-format constraints directly to your prompt.
or
- Provide a few-shot example showing the desired format. (We will try this next).

In [6]:
# Few-shot

prompt = """Summarize the feedback following the example.

–Example 1–

Complaint:
The hotel room was clean, and the staff were friendly.
However, check-in took almost 40 minutes.
The room was also not ready at the promised time.
I had to wait in the lobby with my luggage.
The hotel should improve its check-in process.

Summary:
Positive: Clean room and friendly staff.

Issue: Long check-in time and delayed room availability.

Suggestion: Improve the check-in and room preparation process.

–Example 2–

Complaint:
The breakfast had a good variety of food.
However, several dishes were already cold.
The staff took a long time to refill empty items.
There were also not enough tables during busy hours.
The hotel should improve its breakfast service.

Summary:
Positive: Good variety of breakfast options.

Issue: Cold food, slow refills, and insufficient seating.

Suggestion: Improve food temperature, refill speed, and seating availability.

–Now–
Complaint:
The room was comfortable and had a good view.
However, the air conditioner was very noisy at night.
I contacted reception, but no technician arrived.
This made it difficult for me to sleep.
The hotel should respond to maintenance requests more quickly.
Summary:
"""

In [7]:
few_shot = llm.invoke(prompt)
display(Markdown(few_shot.content))

Positive: Comfortable room and good view.

Issue: Noisy air conditioner and lack of response from maintenance.

Suggestion: Improve the speed of response to maintenance requests.

# Chain of Thought Prompting

**Example: Marketing Strategy Planning**

In [8]:
input_text = """
Should we spend the entire 500,000 THB marketing budget on Facebook Ads,
or should we distribute it across TikTok and Google Search as well?
Come up with an analysis and diversified plan suggestion.
"""

**1.) Without chain-of-thought**

In [9]:
prompt = f"""
{input_text}

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Without Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Without Chain-of-Thought

Spending the entire 500,000 THB on Facebook Ads is risky. While Facebook offers precise targeting, relying on one platform limits your reach and increases vulnerability to algorithm changes or rising ad costs. A diversified strategy is smarter.

Here is a suggested plan:

1.  **Facebook/Instagram (40% - 200,000 THB):** Use this for retargeting and detailed demographic targeting. It’s great for converting users who already know your brand.
2.  **TikTok (30% - 150,000 THB):** Ideal for brand awareness and reaching younger audiences. Short, engaging videos can go viral, lowering your cost per view and building buzz.
3.  **Google Search (30% - 150,000 THB):** Capture high-intent customers who are actively searching for your product. This ensures you catch people ready to buy, not just browsing.

**Why this works:** Facebook builds recognition, TikTok creates excitement, and Google captures immediate sales. This mix balances brand building with direct revenue. It also spreads risk; if one platform underperforms, the others can compensate. Start with this split, monitor results weekly, and shift funds toward the best-performing channel. This approach maximizes your budget’s impact, ensuring you reach customers at every stage of their journey, from discovery to purchase.

**Next, Zero-shot Chain-of-Thought :** `Let's think step-by-step`

In [10]:
prompt = f"""
{input_text}
Let's think step-by-step.
Return the output within 200 words.
Explain in simple language.
"""

ai_msg_no_reasoning = llm.invoke(prompt)

display(Markdown("## Zero-shot Chain-of-Thought"))
display(Markdown(ai_msg_no_reasoning.content))

## Zero-shot Chain-of-Thought

Spending the entire 500,000 THB budget solely on Facebook Ads is risky. While Facebook is great for targeting specific demographics, relying on one platform limits your reach and increases dependency on a single algorithm. A diversified approach is smarter.

Here is a suggested plan:

1.  **Facebook (40% - 200,000 THB):** Use this for retargeting interested users and detailed demographic targeting. It remains strong for conversion and brand awareness among older demographics.
2.  **TikTok (35% - 175,000 THB):** Allocate this to capture younger audiences and drive viral brand awareness. TikTok’s algorithm excels at organic reach and engaging short-form video content, offering high potential for rapid growth.
3.  **Google Search (25% - 125,000 THB):** Use this to capture high-intent users actively searching for your products or services. This ensures you don’t miss customers ready to buy now.

This mix balances awareness (TikTok), engagement (Facebook), and direct demand (Google). It reduces risk if one platform’s costs rise or performance drops. Test each channel for two weeks, then adjust based on cost-per-acquisition data. Diversification ensures broader market coverage and more stable results than putting all eggs in one basket.

> Note: **Nowadays, many models use internal or implicit chain-of-thought reasoning by default**, so simply adding a ```zero-shot chain-of-thought``` prompt **may not** make a significant difference.

**2.) With your chain-of-thought**

> Thinking about :
- Product → Target Customer → Marketing Funnel → Budget Allocation → Risks → Final Recommendation

The answer will be improved.

In [11]:
prompt_reasoning = f"""
{input_text}

Before answering, reason through the following steps:

1. Identify the product/service and target customers.
   - If this information is unknown, assume two clearly different cases
     and provide separate recommendations.

2. Consider which stage of the marketing funnel each channel is best suited for:
   - Awareness
   - Consideration
   - Conversion

3. Estimate how the 500,000 THB budget could be allocated across channels.
   Consider whether each allocation is sufficient for test-and-learn.

4. Analyze the risks of each strategy.

5. Provide a final recommendation with an approximate budget allocation.

Return the output within 200 words.
Explain in simple language.
"""

ai_msg_reasoning = llm.invoke(prompt_reasoning)

display(Markdown("## With Chain-of-Thought"))
display(Markdown(ai_msg_reasoning.content))

## With Chain-of-Thought

Without specific product details, let’s assume two common scenarios: B2C retail (e.g., fashion) and high-intent services (e.g., clinics).

Social media like Facebook and TikTok excel at **Awareness** and **Consideration**. They build brand visibility and engage users through visual storytelling. Google Search dominates **Conversion** by capturing users actively searching for solutions. Spending the entire 500,000 THB on Facebook risks missing high-intent buyers and creates dependency on one platform’s algorithm. Conversely, ignoring social media limits brand reach.

A diversified approach is safer. For B2C retail, allocate 40% to Facebook/Instagram for broad reach, 30% to TikTok for viral potential among younger demographics, and 30% to Google Search to capture ready-to-buy customers. For high-intent services, shift to 20% social for awareness and 80% Google Search, as users there have higher purchase intent.

Risks of single-channel spending include ad fatigue, rising costs, and platform dependency. Diversification allows test-and-learn: you can identify which channel delivers the best return on investment (ROI) and adjust accordingly. With 500,000 THB, each channel gets enough budget to gather meaningful data within a month.

Final recommendation: Do not spend everything on Facebook. Adopt a mixed strategy. For most businesses, start with 50% Facebook/Instagram, 20% TikTok, and 30% Google Search. This balances brand building with direct sales. Monitor performance weekly, pausing underperforming ads and reallocating funds to top-performing channels. This flexible approach minimizes risk and maximizes overall marketing impact.

You can see this **difference** in the response:

`Since your product isn’t specified, let’s look at two common scenarios.`

- **Before CoT**: The **model mainly thinks about how to diversify** the budget.
- **After CoT**: The** model first considers the product **and target customer, **makes assumptions when needed**, then **analyzes the options** and gives a recommendation.

This **improves the depth and structure of the analysis**.

**3.) Few-shot Chain-of-Thought (Geometry Problem)**
> In cases where we want the reasoning steps to follow a clear structure — instead of letting the model generate its own free-form chain-of-thought — we can provide a few labeled examples to guide the reasoning process.
This technique is called few-shot chain-of-thought (few-shot CoT).

In [12]:
few_shot_cot_prompt = """Solve the following math problems step by step:

Example 1:
Problem: If a train travels at 60 miles per hour for 2.5 hours, how far does it go?
Solution:
1. Identify the given information:
   - Speed of the train: 60 miles per hour
   - Time of travel: 2.5 hours
2. Use the formula: Distance = Speed × Time
3. Plug in the values:
   Distance = 60 miles/hour × 2.5 hours
4. Calculate:
   Distance = 150 miles
Therefore, the train travels 150 miles.

Problem: A bakery sold 136 cakes last week. This week, they sold 25% more. How many cakes did they sell this week?
Solution:
1. Identify the given information:
   - Last week's sales: 136 cakes
   - Increase: 25%
2. Calculate the increase:
   25% of 136 = 0.25 × 136 = 34 cakes
3. Add the increase to last week's sales:
   This week's sales = 136 + 34 = 170 cakes
Therefore, the bakery sold 170 cakes this week.

Here is a new question:
Problem: If a rectangle has a length of 15 meters and a width of 8 meters, what is its area?
Solution:
"""

In [13]:
ans = llm.invoke(few_shot_cot_prompt).content
display(Markdown(ans))

Solution:
1. Identify the given information:
   - Length of the rectangle: 15 meters
   - Width of the rectangle: 8 meters
2. Use the formula: Area = Length × Width
3. Plug in the values:
   Area = 15 meters × 8 meters
4. Calculate:
   Area = 120 square meters
Therefore, the area of the rectangle is 120 square meters.

# Additional Topics:

## Analogy-based Prompting

Concept:
> **Let the model recall similar problems from other contexts or domains**, then **transfer useful patterns or lessons** to the current problem.

Expectation:
> **Quickly improve response quality—such as specificity, groundedness, or idea interestingness**—without requiring heavy prompt engineering, such as manually designing detailed reasoning steps or preparing few-shot examples.

**Ex. Employee Adoption of an Internal AI Tool**

In [14]:
problem = """
A company has introduced a new internal AI assistant for employees.

Most employees try the tool once or twice, but then stop using it.
The company has already provided basic training, but adoption remains low.

What should the company do to increase long-term usage?
"""

**Without analogy-based prompting**

In [15]:
prompt = f"""
Problem:
{problem}

Recommend a practical solution.
Keep the response concise.
"""

res = llm.invoke(prompt)

display(Markdown("## Without Analogy-Based Prompting"))
display(Markdown(res.content))

## Without Analogy-Based Prompting

Implement a **"Champion Program”** paired with **context-specific integration**.

1.  **Identify Power Users**: Recruit enthusiastic early adopters from different departments to act as peer coaches. They can demonstrate real-world, role-specific use cases that generic training missed.
2.  **Embed in Workflow**: Integrate the AI directly into existing tools (e.g., Slack, email, CRM) rather than requiring employees to open a separate interface. Trigger prompts should appear at natural decision points (e.g., “Draft this response?” when composing an email).

**Why this works**: Peer influence builds trust, and reducing friction by embedding the tool into daily habits ensures it becomes indispensable rather than optional.

**With analogy-based prompting**

In [16]:
prompt = f"""
Problem:
{problem}

Before solving the problem:

1. Recall 2 similar situations from other domains where people
   tried a new product or behavior but failed to continue using it.

2. Explain what helped improve long-term adoption in those situations.

3. Identify which lessons can be transferred to this workplace problem.

4. Apply those lessons to recommend a practical strategy
   for increasing long-term AI tool usage.

Keep the response concise.

Relevant analogies:

Final recommendation:
"""

ai_msg_with = llm.invoke(prompt)

display(Markdown("## With Analogy-Based Prompting"))
display(Markdown(ai_msg_with.content))

## With Analogy-Based Prompting

### 1. Relevant Analogies

*   **Fitness Apps:** Users download apps after New Year’s resolutions, use them for a few weeks, but quit because the tasks feel like chores rather than helpful habits.
*   **Enterprise Software (e.g., Slack/Teams initially):** Employees resist switching from email because the new tool adds cognitive load without immediate, obvious personal benefit.

### 2. What Helped Improve Adoption

*   **Fitness Apps:** Success came from integrating the app into existing routines (habit stacking), providing immediate micro-rewards (streaks, badges), and lowering the barrier to entry (very short, easy workouts).
*   **Enterprise Software:** Adoption increased when leadership mandated usage for specific low-stakes communications, provided "power user" champions within teams, and demonstrated clear time-saving benefits in daily workflows.

### 3. Transferable Lessons

*   **Integration over Addition:** The tool must fit into existing workflows, not create new ones.
*   **Immediate Value:** Users need to see a quick, tangible win (time saved or error reduced) in the first few interactions.
*   **Social Proof & Champions:** Peer influence and visible usage by respected colleagues drive sustained adoption better than top-down mandates.

### 4. Final Recommendation

**Implement a "Champion-Led Workflow Integration" Strategy:**

1.  **Identify & Empower Champions:** Recruit influential employees from different departments to become AI advocates. Give them advanced training and incentives to share success stories.
2.  **Embed into High-Frequency Tasks:** Instead of generic training, create specific, step-by-step guides for 2-3 common, tedious tasks (e.g., summarizing long emails, drafting routine reports). Make the AI the *easiest* way to complete these specific tasks.
3.  **Gamify Early Wins:** Launch a short-term challenge where employees earn recognition for sharing how the AI saved them time. Highlight these wins in company communications to build social proof.
4.  **Reduce Friction:** Ensure the AI is deeply integrated into existing tools (e.g., inside Microsoft Word or Slack) so employees don’t have to switch contexts to use it.

- **Without Analogy**: More generic and less interesting recommendations, based mainly on the immediate problem.
- **With Analogy**: Encourages the model to think through similar cases, which can lead to more interesting, specific, and grounded ideas.

Helpful Website for Prompting Techniques : https://www.promptingguide.ai/techniques

## LLM-as-Judge

**1. Prepare the ideas to evaluate.**

In [17]:
import pandas as pd
import json

# Ideas to evaluate
ideas = {
    "Idea 1": """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
""",

    "Idea 2": """
Develop an automated OCR system for processing invoice forms
for the Accounting Department.

The goal is to reduce manual data-entry time by 40%
and reduce the data error rate to below 2% within this quarter.

The project will run as a 3-week pilot using
1 developer and 1 accounting staff member
before full deployment.
"""
}

**2. Define the Evaluation Rubric**
> Using a detailed scoring rubric with clearly specified criteria is recommended to improve the consistency of the LLM judge.

In [18]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**3. Run the LLM-as-Judge Loop**

In [19]:
results = []

for idea_name, idea in ideas.items():

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON in this format:

{{
  "clarity": 1,
  "clarity_rationale": "...",
  "business_value": 1,
  "business_value_rationale": "...",
  "feasibility": 1,
  "feasibility_rationale": "...",
  "final_suggestion": "..."
}}
"""

    response = llm.invoke(prompt).content.strip()

    # Remove Markdown code fences if the model adds them
    response = response.replace("```json", "").replace("```", "").strip()

    result = json.loads(response)

    results.append({
        "Idea": idea_name,
        "Clarity": result.get("clarity"),
        "Business Value": result.get("business_value"),
        "Feasibility": result.get("feasibility"),
        "Clarity Rationale": result.get("clarity_rationale", ""),
        "Business Value Rationale": result.get("business_value_rationale", ""),
        "Feasibility Rationale": result.get("feasibility_rationale", ""),
        "Final Suggestion": result.get("final_suggestion", "N/A")
    })

df = pd.DataFrame(results)
display(df)

,Idea,Clarity,Business Value,Feasibility,Clarity Rationale,Business Value Rationale,Feasibility Rationale,Final Suggestion
0,Idea 1,3,3,2,The core concept of using AI for document proc...,There is good potential impact regarding cost ...,The proposal suggests using free open-source t...,Refine the proposal by defining specific high-...
1,Idea 2,4,4,2,The idea is understandable and identifies a sp...,There is clear business alignment with efficie...,"While steps are proposed (3-week pilot), there...",Extend the pilot timeline to at least 8-12 wee...


**Designing LLM-as-Judge is iterative refinement work, you need to inspect the prompt <-> response of the judgement , calibrate the rubric to better align your task**



## Iterative Refinement

> Reflection Loop: We can integrate LLM-as-a-Judge into a refinement loop to automatically improve response quality based on the judge’s feedback.

Idea → Judge → Feedback → Refine → Re-evaluate → Compare Before vs. After

In [20]:
# Original low-scoring idea
idea_1 = """
Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.
"""

**Detailed Rubric 1 to 5**

In [21]:
rubric = """
### Clarity
- Score 1: Vague or confusing; no clear problem or target audience.
- Score 2: Broad idea, but lacks a defined problem and specific audience.
- Score 3: Understandable, but lacks important details or focus.
- Score 4: Clear problem and audience, with minor gaps in structure or focus.
- Score 5: Extremely clear, specific, and well-structured.

### Business Value
- Score 1: No clear business benefit.
- Score 2: Potential benefit, but weak connection to business goals.
- Score 3: Good potential impact, but lacks measurable KPIs.
- Score 4: Clear business alignment and measurable KPIs,
  but ROI or concrete impact is still incomplete.
- Score 5: Strong business alignment, measurable KPIs,
  and clear ROI or business impact.

### Feasibility
- Score 1: Unrealistic and lacks actionable steps.
- Score 2: Some steps are proposed, but major resource or execution issues remain.
- Score 3: Actionable, but missing important details such as who, when, or how.
- Score 4: Clear execution plan with minor gaps in resources or timeline.
- Score 5: Highly actionable with clear steps,
  realistic resources, ownership, and timeline.
"""

**Judge Prompt Function**

In [22]:
def judge_idea(idea):

    prompt = f"""
You are a strict but fair Business Strategy Director.

Evaluate the following business idea using the rubric below.

{rubric}

Idea:
{idea}

Return ONLY valid JSON:

{{
  "clarity": 1,
  "business_value": 1,
  "feasibility": 1,
  "feedback": "Give concise and actionable feedback for improving the idea."
}}
"""

    response = llm.invoke(prompt).content.strip()

    response = (
        response
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    return json.loads(response)

**Run the Reflection Loop**

In [23]:
from tqdm.notebook import tqdm

steps = [
    "Evaluate original idea",
    "Reflect and improve",
    "Evaluate improved idea"
]

with tqdm(total=len(steps), desc="Reflection Loop") as pbar:

    # Step 1: Evaluate the original idea
    before = judge_idea(idea_1)
    pbar.update(1)

    # Step 2: Improve the idea using judge feedback
    refinement_prompt = f"""
Original Idea:
{idea_1}

Evaluator Feedback:
{before["feedback"]}

Revise the idea to address the feedback.

Improve:
- clarity and scope
- measurable business value
- feasibility and execution details

Do not change the core objective of using AI for document processing.

Return only the improved idea.
"""

    improved_idea = llm.invoke(refinement_prompt).content.strip()
    pbar.update(1)

    # Step 3: Evaluate the improved idea
    after = judge_idea(improved_idea)
    pbar.update(1)

Reflection Loop:   0%|          | 0/3 [00:00<?, ?it/s]

**Shows the idea : before vs. after**

In [24]:
comparison = pd.DataFrame([
    {
        "Version": "Before",
        "Clarity": before["clarity"],
        "Business Value": before["business_value"],
        "Feasibility": before["feasibility"],
    },
    {
        "Version": "After",
        "Clarity": after["clarity"],
        "Business Value": after["business_value"],
        "Feasibility": after["feasibility"],
    }
])

comparison["Average"] = comparison[
    ["Clarity", "Business Value", "Feasibility"]
].mean(axis=1)

display(comparison)

,Version,Clarity,Business Value,Feasibility,Average
0,Before,2,3,2,2.333333
1,After,5,5,5,5.000000


In [25]:
# Show the ideas and judge feedback

display(Markdown("## Before Refinement"))
display(Markdown(idea_1))

display(Markdown("### Judge Feedback"))
display(Markdown(before["feedback"]))

## Before Refinement


Introduce AI technology to automatically read and process
documents and forms across the organization.

The goal is to reduce manual data-entry workload,
improve overall operational speed, and significantly reduce costs.

The development team will immediately experiment with free
open-source tools so that the project can deliver results quickly.


### Judge Feedback

Define the specific document types and target departments to narrow the scope. Replace vague goals with quantifiable KPIs (e.g., 'reduce processing time by 40%'). Address the risks of using unvetted open-source tools in a production environment and establish a formal roadmap with assigned ownership and timeline rather than an immediate, unstructured experiment.

In [26]:
display(Markdown("## After Refinement"))
display(Markdown(improved_idea))

display(Markdown("### Judge Feedback"))
display(Markdown(after["feedback"]))

## After Refinement

**Project Title: Targeted AI Document Automation Pilot for Finance and HR Departments**

**Objective:**
Implement a secure, enterprise-grade AI document processing solution to automate data extraction from high-volume forms, specifically targeting the Finance (Invoice Processing) and Human Resources (Onboarding Documents) departments.

**Scope:**
1.  **Target Departments:** Finance and HR.
2.  **Document Types:** Standardized vendor invoices and new hire onboarding packets (ID verification, tax forms).
3.  **Exclusions:** Unstructured contracts and legal reviews are excluded from this initial phase to ensure manageable complexity.

**Measurable Business Value (KPIs):**
*   **Efficiency:** Reduce average document processing time by **40%** (from 15 minutes to 9 minutes per document).
*   **Accuracy:** Achieve a data extraction accuracy rate of **95%** or higher, reducing manual correction efforts.
*   **Cost:** Decrease manual labor costs associated with data entry by **20%** within the first six months of deployment.

**Feasibility and Execution Plan:**
1.  **Technology Strategy:** Instead of relying on unvetted open-source tools, the team will conduct a rapid evaluation of three enterprise-ready AI platforms (e.g., AWS Textract, Azure Form Recognizer, or Google Document AI) that offer built-in security compliance (GDPR/HIPAA) and support. A proof-of-concept (PoC) will be run for two weeks to compare performance against the defined KPIs before selecting a vendor.
2.  **Roadmap & Ownership:**
    *   **Phase 1 (Weeks 1-4):** Requirements gathering and vendor selection, led by the IT Solutions Architect.
    *   **Phase 2 (Weeks 5-8):** PoC development and integration with existing ERP/HR systems, led by the AI Engineering Lead.
    *   **Phase 3 (Weeks 9-12):** Pilot launch with Finance and HR teams, including user training and feedback loops, managed by the Change Management Specialist.
    *   **Phase 4 (Week 13+):** Full rollout and continuous optimization.
3.  **Risk Mitigation:** All data will remain within secure, compliant cloud environments. A human-in-the-loop verification step will remain in place during the pilot phase to ensure quality and build user trust.

### Judge Feedback

This is a highly structured and professional proposal. To ensure success, explicitly define the 'Human-in-the-loop' workflow in Phase 3 to quantify the remaining manual effort vs. automated effort. Additionally, consider adding a specific budget estimate for the vendor PoC and licensing costs to strengthen the ROI calculation.

Now, the LLM can automatically optimize the idea against the given rubric through a closed-loop refinement process.